# 📘 目錄

1. [摘要](#摘要)
2. [馬可夫決策框架](#馬可夫決策框架)
3. [賣股問題模型設計](#賣股問題模型設計)
4. [強化學習策略設計](#強化學習策略設計)
5. [實戰](#實戰)
6. [結論與未來展望](#結論與未來展望)
7. [參考文獻](#參考文獻)


<span style='font-size:16px; color:;'>  
    
## 一. 摘要  <a id="摘要"></a>
    
股票市場有一句名言「會買的是徒弟，會賣的才是師父」，賣股的時機一直是讓投資人頭痛的問題。為了解決這個難題，本研究以量化的角度將其包裝成馬可夫決策過程（Markov Decision Process, MDP）。透過各種的強化學習方法(Reinforcement Learning policy)： SARSA、Q-learning、SARSA($\lambda$)，設計出賣股策略，目標是提升投資績效。

    
    
**最佳賣點問題：**
1. 問題建模 (Problem Formulation)：   
    將股票交易中的賣出時機選擇問題，正式定義為一個馬可夫決策過程 (MDP)，使決策過程具備數學上的嚴謹性。  
<br>    
    
2. 演算法應用 (Methodology)：   
    採用強化學習 (Reinforcement Learning) 中的 SARSA 、Q-learning 以及 SARSA($\lambda$) 演算法，訓練代理人 (Agent) 學習在動態市場環境下的最佳決策。
<br>
    
3. 最佳賣點任務 (Task Definition)：  
    * 限制條件：投資人必須在未來 $T$ 天的時間窗口內，賣出手中的一支股票。(最晚會在第$T$天強制賣出)
    * 報酬計算：考量「機會成本」與「時間價值」，將賣股所得與後續存入銀行的利息總和納入考量。  

    <br>
    
4. 研究目標 (Objective)：  
    比較不同策略的表現，旨在設計出一套能顯著提升投資績效的最佳賣股策略。

<span style='font-size:16px; color:;'>  
    
## 二. 馬可夫決策框架  <a id="馬可夫決策框架"></a>
    
參考Powell(2022)，馬可夫決策過程(MDP)有五個核心的要素(Powell Framework)，分別是狀態變數、決策變數、外生資訊、狀態轉移函數以及目標函數，以下分別介紹基本概念。
    

### 1. 狀態變數（State Variables, $S_t$）
    
狀態變數代表了在時間點 $t$ 做決策時所擁有的「所有知識與資訊」的集合。根據 Powell 的定義，狀態變數必須具備馬可夫性質（Markov Property），即包含所有計算決策及推導下一個狀態所需的必要資訊。$S_t$ 通常包含以下三類資訊：物理狀態（Physical State）：例如庫存量、位置、資金餘額等具體資源。資訊狀態（Information State）：例如當前的市場價格、天氣數據等外在環境數據。信念狀態（Belief State）：當系統包含不確定參數時（例如未知的需求分佈參數），我們對這些參數的主觀機率分佈或估計值。
    
    
### 2. 決策變數（Decision Variables, $x_t$）
    
決策變數是在時間點 $t$，根據當前狀態 $S_t$ 所採取的行動或控制量。在強化學習中常被稱為 Action ($a_t$)。決策必須受到限制條件（Constraints）的約束，我們通常用 $\mathcal{X}_t$ 來表示在時間 $t$ 的可行決策集合，因此：$$x_t \in \mathcal{X}_t$$決策的產生通常依賴於一個策略（Policy），記為 $\pi$，即 $x_t = X^\pi(S_t)$。我們的目標就是找到一個表現最好的函數 $\pi$。
    
### 3. 外生資訊（Exogenous Information, $W_{t+1}$）
這是在我們做出決策 $x_t$ 之後，但在時間 $t+1$ 之前首次得知的新資訊。這是系統隨機性（Stochasticity）的來源。這些資訊可能包括：隨機的需求變化。資產價格的波動。系統的隨機故障或延遲。在數學上，我們通常假設 $W_{t+1}$ 是一個隨機變數，其分佈可能依賴於當前的狀態 $S_t$。
    
### 4. 狀態轉移函數（Transition Function, $S^M$）
狀態轉移函數描述了系統如何從時間 $t$ 演變到時間 $t+1$。它是連結狀態、決策與外生資訊的數學模型。不同於傳統 MDP 教科書有時使用轉移矩陣（Transition Matrix），Powell 架構強調使用顯式的方程形式：$$S_{t+1} = S^M(S_t, x_t, W_{t+1})$$這個函數 $S^M(\cdot)$ 包含了所有的物理守恆定律、邏輯規則或市場演變公式。例如：庫存更新公式 $R_{t+1} = R_t - x_t + \hat{D}_{t+1}$（其中 $\hat{D}$ 為隨機需求）。
    
### 5. 目標函數（Objective Function）
目標函數用來評估我們決策的好壞。通常由兩部分組成：貢獻/成本函數（Contribution/Cost Function, $C(S_t, x_t)$）：單期決策帶來的立即收益或成本。期望值（Expectation）：因為未來充滿不確定性（由 $W$ 造成），我們追求的是最大化累積獎勵的期望值。數學上，我們的目標是尋找最佳策略 $\pi^*$ 來最大化總價值：$$\max_{\pi} \mathbb{E} \left[ \sum_{t=0}^{T} \gamma^t C(S_t, X^\pi(S_t)) \right]$$其中 $\gamma$ 為折現因子（Discount Factor）。
    

<span style='font-size:16px; color:;'>  

## 三. 賣股問題模型設計 <a id="賣股問題模型設計"></a>
本研究在交易策略設計上參考強化學習於金融交易之應用文獻，特別是比較 Q-learning 與 SARSA 的表現（Corazza & Sangalli, 2015），並結合隨機最佳化與序列決策之理論架構（Powell, 2022），將「股票最佳賣出時點選擇」問題建構為一個有限期時序決策問題(Finite-horizon sequential decision-making problem)，以下為設計的的Powell Framework
### 1. 狀態變數（State Variables, $S_t$）
$$
S_t = (e_{t-\ell},e_{t-(\ell-1)},...,e_{t-1},e_{t},M_t)
$$
* $t$: 目前的時間期數，$t=0 ～ T$
* $e_t$: $t$時間點的對數報酬(log return)，$e_t = \log \frac{P_t}{P_{t-1}}$
* $M_t$: $t$時間點時相對於期初的對數報酬，$M_t = \log \frac{P_t}{P_0}$
* 賣出的話，$S_t=\emptyset$(terminal state)
    
如果是均值回歸的信者，當發現過去一段時間股價報酬一直創新高會考慮賣股；趨勢交易者發現股價向下突破某一閥值也會考慮賣股。狀態變數的設計考慮了這兩種情況，因此將一段時間之前的對數報酬(近期報酬)以及目前的總報酬率(長期報酬)考慮進來，期望擁有更全面的判斷能力。
#### 離散化方法：分位數分箱 (Quantile Binning)
由於$e_t$和$M_t$報酬率都是連續變數，對於建立Q-table非常不利，因此我們需要離散化狀態變數。  
* 邏輯：將歷史(模擬)資料(單期報酬、累積報酬)切成 n 等份，再把當前的報酬分配到對應的區間內。   
* 特色：不管資料分佈多麼不均勻（例如極端值很多），每個箱子裡的資料筆數是一樣多的，能夠適應不同股價的性質。  
* 例子：假設 $n_1=4$ (把單期報酬切成 4 等份)，它會找出 25%、50%、75% 的位置當作切點 (Cut-offs)，$n_2=5$(把累積報酬切成5等分)，考慮2天的近期報酬。則狀態數目就只剩下$5*4^2$
    
    
### 2. 決策變數（Decision Variables, $x_t$）
在尚未賣出且未達到投資期限的情況下，投資者可以選擇明天($t+1$)要賣出(Sell)或是繼續
持有(Hold)，但在最後一天$T$時，必須強制賣出。在任何時間點賣出，系統都會進入終止狀態(terminal state)
    
$$
x_t = X_t(S_t) = 
\begin{cases} 
\{\text{Hold, Sell}\} ,&  x_{t-1}=\text{Hold}\\
\text{Sell} ,& t=T-1 \\
\end{cases}
$$


### 3. 外生資訊（Exogenous Information, $W_{t+1}$）
外生資訊定義為驅動股價變動的隨機變數，可以來自於布朗運動(BM)、碎形布朗運動(fBM)、Heston Model等。  
定義 $t+1$ 時刻的外生資訊向量為：
$$
W_{t+1} = Z_{t+1}
$$
    
假如選用BM於模擬環境中，$Z_{t+1}$就是標準常態分配變數，即$Z_{t+1}\sim N(0,1)$
    
### 4. 狀態轉移函數（Transition Function, $S^M$）
我們的狀態是由股價來計算組成，因此股價狀態的更新能夠代表狀態轉移函數。針對不同的價格隨機模型，股價的轉移方程也有所不同：
    
$$
P_{t+1} = f(P_t)
$$
* $f()$：股價隨機過程，例如BM：
    
$$P_{t+1} = P_t \cdot \exp\left( \left(\mu - \frac{\sigma^2}{2}\right)\Delta t + \sigma \sqrt{\Delta t} Z_{t+1} \right)$$
其中，$\mu$：預期報酬率 (Drift)  $\sigma$：波動率 (Volatility)  $\Delta t$：時間間隔  $Z_{t+1}$：來自外生資訊 $W_{t+1}$ 的隨機項  
    
配合前述的狀態變數 $S_t$，其餘變數（如單期報酬$e_t$、累積報酬 $M_t$）亦隨之更新：

$$
S_{t+1} = 
\begin{cases}
M_{t+1}=\log \frac{P_{t+1}}{P_0}, e_{t+1}=\log \frac{P_{t+1}}{P_t} & x_t=\text{Hold}\\
\emptyset, & x_t=\text{Sell}
\end{cases}
$$
        
### 5. 目標函數（Objective Function）
本研究的目標是在有限期限 $T$ 天內，選擇最佳賣出時點，讓期望累積報酬率最大化。
* 選擇持有，則當期不產生即時報酬
* 如果在第$\tau$天賣出，所得到的現金$P_\tau$會投入銀行並以無風險利率$r_f$滾存到第$T$天
    
**Contibution Function(累積報酬率)**
$$
C(S_t,x_t)=
\begin{cases}
0, & x_t=\text{Hold}\\
\frac{P_{t+1}(1+r_f)^{T-t-1}}{P_0} - 1, & x_t=\text{Sell}
\end{cases}
$$
    
**最佳策略$\pi^*$**
$$
\max_{\pi} \mathbb{E} \left[ \sum_{t=0}^{T} C(S_t, X^\pi(S_t)) \right]
$$
由於任務定義已經考慮了無風險利率報酬的時間成本，所以目標函數不再設計折現率$\gamma$。

<span style='font-size:16px; color:;'>  
    
## 四. 強化學習策略設計  <a id="強化學習策略設計"></a>
    
### 1. SARSA
SARSA 為一種在策略（On-policy）的時間差分（Temporal-Difference, 
TD）學習演算法。其核心特點在於：代理人（agent）在更新狀態–動作價值時，必須依據實際所採取的動作進行更新，而非假設未來一定會選擇最佳動作。SARSA 在學習過程中，行為策略（behavior policy）與被評估的策略（target policy）是相同的。代理人根據當前策略（通常為ε-greedy）選擇動作，並且在更新 Q 值時，使用「下一狀態中實際選擇的動作」所對應的 Q 值進行更新。因此，SARSA 能夠忠實反映探索行為對學習過程的影響。
    
#### SARSA更新公式 (使用實際採取的 $x_{t+1}$）：
$Q(s_t,x_t)=Q(s_t,x_t) + \alpha [r_{t+1}+\gamma Q(s_{t+1},x_{t+1})-Q(s_t,x_t)]$  
*註：這裡的 $x_{t+1}$ 是代理人實際會去執行的動作。*
    
**最佳賣點任務的SARSA更新規則可以分成兩種情況：** 
    
情況 A：當 $x_t = \text{Hold}$  
    
$$Q(s_t, \text{Hold}) \leftarrow Q(s_t, \text{Hold}) + \alpha [0 + 1 \cdot Q(s_{t+1}, x_{t+1}) - Q(s_t, \text{Hold})]$$
    
(這裡 $\gamma=1$配合contribution function $C$ 的設計，獎勵是 0)  
    
情況 B：當 $x_t = \text{Sell}$
    
$$Q(s_t, \text{Sell}) \leftarrow Q(s_t, \text{Sell}) + \alpha [C(S_t,\text{Sell}) + 0 - Q(s_t, \text{Sell})]$$
    
(因為賣出後遊戲結束，沒有下一期的 Q 值)
 
其中：  
* $\alpha\in (0,1]$：學習率（learning rate），控制每次更新對舊 Q 值的影響程度 
* $Q(s,x)$：狀態–動作對應的 Q 值
    
### 2. Q-learning
Q-Learning 為一種離策略（Off-policy）學習演算法。其特點在於：代理人（agent）在更新狀態–動作價值時，不依賴實際執行的動作，而是根據「在下一個狀態中能使𝑄值最大的動作」來進行更新。  
換言之，Q-Learning在學習過程中，行為策略（behavior policy）與被評估
的策略（target policy）是不同的。即使代理人因探索而執行了非最佳動作，更新時仍假設未來會選擇最佳行動。
    
#### Q-learning更新公式（使用理論最佳的 $\max x'$）：
$Q(s_t,x_t)=Q(s_t,x_t) + \alpha  [r_{t+1}+\gamma \max\limits_{x'} Q(s_{t+1},x')-Q(s_t,x_t)]$  
*註：這裡不管代理人實際上會做什麼，直接取最大值來更新。*

**最佳賣點任務的Q-learning更新規則也可以分成兩種情況：** 
    
情況 A：當 $x_t = \text{Hold}$  
    
$$Q(s_t, \text{Hold}) \leftarrow Q(s_t, \text{Hold}) + \alpha [0 + 1 \cdot \max\limits_{x'} Q(s_{t+1}, x') - Q(s_t, \text{Hold})]$$
    
    
情況 B：當 $x_t = \text{Sell}$
    
$$Q(s_t, \text{Sell}) \leftarrow Q(s_t, \text{Sell}) + \alpha [C(S_t,\text{Sell}) + 0 - Q(s_t, \text{Sell})]$$
    

   
### 3. SARSA($\lambda$)
SARSA($\lambda$) 是 SARSA 的改進版本，它引入了資格跡（Eligibility Traces, $E_t$） 的概念。傳統的 SARSA (即 $\lambda=0$) 每次只更新前一步的 Q 值，屬於「短視」的更新；而 SARSA($\lambda$) 則允許獎勵向後傳遞給過去一連串導致當前狀態的動作。  
<br>
在我們的賣股任務中，這一點特別重要。假設代理人連續「持有」了 20 天，並在第 21 天「賣出」獲得高額報酬。傳統 SARSA 需要重複經歷非常多次這個過程，才能慢慢將第 21 天的獎勵一步步傳回第 1 天；而 SARSA($\lambda$) 能透過資格跡，在一次賣出後，立即更新過去這 20 天所有「持有」決策的價值，大幅提升學習效率。    

**核心機制：資格跡 (Eligibility Traces)**  
我們為每一個狀態–動作對 $(s,x)$ 維護一個變數 $E(s,x)$。  
* 當代理人訪問 $(s_t, x_t)$ 時，$E(s_t, x_t)$ 會增加（通常加 1）。
* 隨著時間推移，所有 $E(s,x)$ 會以 $\gamma \lambda$ 的速率衰減。
* 更新時，不只更新當前的 Q 值，而是按 $E(s,x)$ 的比例更新所有走過的狀態。
    
#### SARSA($\lambda$) 更新公式：
首先計算 TD 誤差（TD Error, $\delta_t$）：
    $$\delta_t = r_{t+1} + \gamma Q(s_{t+1}, x_{t+1}) - Q(s_t, x_t)$$
    
接著更新所有狀態的 Q 值與資格跡：
$$E_t(s,x) \leftarrow \gamma \lambda E_{t-1}(s,x) + \mathbb{1}(s=s_t, x=x_t)
$$
    
$$
Q(s,x) \leftarrow Q(s,x) + \alpha \delta_t E_t(s,x)
$$
   
**最佳賣點任務的 SARSA($\lambda$) 更新規則同樣依據是否賣出分為兩種情況：**  
    
情況 A：當 $x_t = \text{Hold}$  
1. 計算誤差：$\delta_t = 0 + 1 \cdot Q(s_{t+1}, x_{t+1}) - Q(s_t, \text{Hold})$
2. 增加當前狀態的資格跡：$E(s_t, \text{Hold}) \leftarrow E(s_t, \text{Hold}) + 1$
3. 更新所有 $(s,x)$ 的 Q 值：$Q(s,x) \leftarrow Q(s,x) + \alpha \delta_t E(s,x)$
4. 衰減所有資格跡：$E(s,x) \leftarrow 1 \cdot \lambda \cdot E(s,x)$
    
情況 B：當 $x_t = \text{Sell}$  
1. 計算誤差：$\delta_t = C(S_t, \text{Sell}) + 0 - Q(s_t, \text{Sell})$ (賣出即終止，未來價值為0)
2. 增加當前狀態的資格跡：$E(s_t, \text{Sell}) \leftarrow E(s_t, \text{Sell}) + 1$
3. 更新所有 $(s,x)$ 的 Q 值：$Q(s,x) \leftarrow Q(s,x) + \alpha \delta_t E(s,x)$
4. 因為 Episode 結束，重置所有 $E(s,x) = 0$，準備下一回合。 
    
其中：  
* $\lambda \in [0, 1]$：衰減因子。$\lambda=0$ 即為傳統 SARSA；$\lambda=1$ 類似於蒙地卡羅方法（Monte Carlo），等到結局才算總帳。
* $E(s,x)$：紀錄某個狀態動作對在近期被訪問的頻率與時間遠近。


<span style='font-size:16px; color:;'>  
    
## 五. 實戰  <a id="實戰"></a>
###  回測環境設定
#### a. 選取資產：上市台股1000檔  
#### b. 最大回測時間：2000-01-04 ～ 2025-11-19  
#### c. 賣出規則：在測試期間的T天內，可以根據至第t天之前的資訊判斷，並且以隔日第t+1天的價格賣出。
#### d. RL方法：QL、SARSA、SARSA(𝜆)，隨機來源使用BM、fBM(-f)。
#### e. 比較對象：基礎策略，B&H、❓、💰；以及技術指標，MA、KD、RSI、MACD、。  
#### f. 回測方法：採用時間序列之拔靴法(bootstrap methods for time series) (Haruiv,2017; Efron & Tibshirani,1993)


**補充**：  
B&H：買入持有，最後一天賣出  
❓：隨機挑一天賣出  
💰：第一天賣出股票存銀行，賺取利息。  
MA：若短天期均線(n=5) < 長天期均線(n=20)，賣出  
KD：若K線 < D線(n=9)，賣出  
RSI： rsi值(n=14) > 70 表示股票過熱，賣出  
MACD： macd柱狀圖(12,26,9) < 0，賣出  

**時間序列之拔靴法**：隨機挑選一支股票的一段連續時間(例如500天)當作訓練資料，並且在接下來的100天(也就是第501~600天)作為測試資料，用以比較各種方法的績效。此方法可在保留時間相依結構的情況下進行重抽樣，能夠降低單一樣本區間可能導致的樣本偏誤，提升績效結果的可靠性。


In [ ]:
import os 
os.chdir(r'C:\Users\ytchen\anaconda3\envs\liu_env\Project')
import numpy as np
import pandas as pd
import random
from packages.psp2 import bm_simulation_single, fbm_simulation_single, hurst_dfa, hm_single, hm_params_final
from tqdm import tqdm
from collections import deque, Counter

# --- 1. 建立切點 (Training階段做一次) ---
def get_bins_quantile(returns, n):
    """
    等量切分n個區塊
    returns: log(pi/p_{i-1}) or log(pi/p0)
    針對「單日報酬」e_t 或 「累積報酬」 M_t計算切點
    每一區數量相同
    """
    percentiles = np.linspace(0, 100, n + 1)
    values = np.percentile(returns, percentiles)
    values = values[1:-1] # 去掉 0% 和 100%
    values = sorted(list(set(values)))
    if len(values)+1 < n:
        print(f'區塊數量不足：{len(values)+1} < {n}')
    return values


def get_bins_std(returns, n):
    """
    等標準差切分n個區塊
    returns: log(pi/p_{i-1}) or log(pi/p0)
    針對「單日報酬」e_t 或 「累積報酬」 M_t計算切點
    每一區塊數量不同
    """
    max_std = 2.0 #設定的最大標準差
    std = np.std(returns)
    mu = np.mean(returns)
    
    percentiles = np.linspace(-max_std, max_std, n + 1)
    values = mu + std * percentiles
    values = values[1:-1] # 去掉 0% 和 100%
    values = sorted(list(set(values)))
    if len(values)+1 < n:
        print(f'區塊數量不足：{len(values)+1} < {n}')
    return values

# --- 2. 狀態編碼 (Step-by-step 執行) ---
def encoding(recent_returns, M_t, bins_return, bins_M):
    """
    將 (e_t序列, M_t) 混合編碼成唯一的 State ID
    
    參數:
    - recent_returns: list or array, 過去幾天的單日報酬 e.g., [0.01, -0.02, ...]
    - M_t: float, 目前的累積報酬
    - bins_return: list, 單日報酬的切點
    - bins_M: list, 累積報酬的切點
    """
    # 1. 處理近期的單日報酬 (Vector)
    recent_returns = np.array(recent_returns)
    # 找出每個 e_t 落在哪個區間 (0 ~ n_bins_ret-1)
    idx_returns = np.searchsorted(bins_return, recent_returns, side='right')
    
    # 2. 處理當前的長期報酬 (Scalar)
    # 找出 M_t 落在哪個區間 (0 ~ n_bins_M-1)
    idx_M = np.searchsorted(bins_M, [M_t], side='right')[0]
    
    # 3. 混合編碼 (Flattening)
    # 概念： ID = (e_t 的組合 ID) + (M_t 的 ID * e_t 組合的總可能性)
    
    # A. 先算 e_t 部分的權重
    n_bins_ret = len(bins_return) + 1  # 單日報酬有幾種狀態
    N = len(recent_returns)            # 看幾天
    weights_ret = n_bins_ret ** np.arange(N)
    
    id_from_returns = np.dot(idx_returns, weights_ret)
    
    # B. 計算 e_t 部分總共有多少種可能 (Offset)
    total_states_returns = n_bins_ret ** N
    
    # C. 加上 M_t 的貢獻
    # 最終 ID = (M_t區間編號 * 前面所有組合的數量) + 前面的組合ID
    final_index = (idx_M * total_states_returns) + id_from_returns
    
    return int(final_index)

def compute_MA(close,fast=5,slow=20):
    ma_fast = close.rolling(fast).mean()
    ma_slow = close.rolling(slow).mean()
    return ma_fast - ma_slow

def compute_KD(close, n=9):
    low_n = close.rolling(window=n, min_periods=n).min()
    high_n = close.rolling(window=n, min_periods=n).max()

    rsv = (close - low_n) / (high_n - low_n) * 100

    K = rsv.ewm(alpha=1/3, adjust=False).mean()
    D = K.ewm(alpha=1/3, adjust=False).mean()

    return K, D
    
def compute_RSI(close, n=14):
    delta = close.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1/n, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/n, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    return rsi


def compute_MACD(close, fast=12, slow=26, signal=9):
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()

    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=signal, adjust=False).mean()
    macd_hist = macd - macd_signal

    return macd, macd_signal, macd_hist




class SmartSelling_env:
    def __init__(self, sim_returns, before, e_num_block=5, M_num_block=10):
        self.sim_returns = sim_returns #log return
        self.before = before # 看最近幾期報酬e_{t-ell}
        self.rf = 0.02 #銀行年利率
        self.dt = 1/252
      
        # self.bins_e = get_bins_quantile(sim_returns.flatten(), e_num_block) #單日報酬的切點
        self.bins_M = get_bins_quantile(np.cumsum(sim_returns,axis=1).flatten(), M_num_block)#累積報酬的切點

        self.bins_e = get_bins_std(sim_returns.flatten(), e_num_block) #單日報酬的切點
        # self.bins_M = get_bins_std(np.cumsum(sim_returns,axis=1).flatten(), M_num_block)#累積報酬的切點
        
        
        self.state_num = e_num_block**before * M_num_block
        self.action_num = 2#1:out, 0:hold
        

    def reset(self,idx):
        self.now_returns = deque(self.sim_returns[idx].flatten())
        self.e_t = deque([self.now_returns.popleft() for _ in range(self.before)],maxlen=self.before)
        self.M_t = np.sum(self.e_t)
        st = encoding(self.e_t, self.M_t, self.bins_e, self.bins_M)
        return st


    def step(self,action):
        self.e_t.append(self.now_returns.popleft())
        self.M_t += self.e_t[-1]
        next_st = encoding(self.e_t, self.M_t, self.bins_e, self.bins_M)

        
        if len(self.now_returns) <= 1:
            done = True
            reward = np.exp(self.M_t + self.now_returns.popleft()) -1#np.exp(self.M_t + self.now_returns[0]):明天賣掉拿到的錢
            
            
        elif action == 1:#out
            done = True
            reward = np.exp(self.M_t + self.now_returns.popleft()) * (1 + len(self.now_returns)*self.dt*self.rf) - 1
            
            
        elif action == 0:#hold:
            done = False
            reward = 0
            
        return next_st, reward, done
      
    def action_sample(self):
        return np.random.randint(0,2)
   
    
    
def q_learning(env, n_episodes=100, alpha=0.1, gamma=1.0, epsilon=0.1, seed=None, verbose=True):

   # TODO: initialization (same as SARSA)
   n_states = env.state_num
   n_actions = env.action_num
   Q = np.zeros((n_states, n_actions))
   # Q_hist = np.empty((0, n_states, n_actions))

   if seed is not None:
       np.random.seed(seed)

   def epsilon_greedy_policy(state):
        # TODO: epsilon-greedy selection
       if np.random.rand() < epsilon:
           return env.action_sample()
       else:
           return np.argmax(Q[state])

   if verbose:
       iter_tool =  tqdm(range(n_episodes),desc='Q-learning')
   else:
       iter_tool =  range(n_episodes)

   for idx in iter_tool:
       state = env.reset(idx)
       action = epsilon_greedy_policy(state)
       done = False

       while not done:
           next_state, reward, done = env.step(action)
           next_action = epsilon_greedy_policy(next_state)

           # TODO: Q-learning update rule
           Q[state, action] += alpha * (reward + gamma * np.max(Q[next_state]) - Q[state, action])

           # move forward
           state = next_state
           action = next_action

       # Q_hist = np.vstack([Q_hist, Q.reshape(1, n_states, n_actions)])
   # return Q_hist
   return Q 
   

def sarsa(env, n_episodes=100, alpha=0.1, gamma=1.0, epsilon=0.1, seed=None, verbose=True):

   # TODO: initialization (same as SARSA)
   n_states = env.state_num
   n_actions = env.action_num
   Q = np.zeros((n_states, n_actions))
   # Q_hist = np.empty((0, n_states, n_actions))

   if seed is not None:
       np.random.seed(seed)

   def epsilon_greedy_policy(state):
        # TODO: epsilon-greedy selection
       if np.random.rand() < epsilon:
           return env.action_sample()
       else:
           return np.argmax(Q[state])

   if verbose:
       iter_tool =  tqdm(range(n_episodes),desc='SARSA')
   else:
       iter_tool =  range(n_episodes)

   for idx in iter_tool:
       state = env.reset(idx)
       action = epsilon_greedy_policy(state)
       done = False

       while not done:
           next_state, reward, done = env.step(action)
           next_action = epsilon_greedy_policy(next_state)

           # TODO: SARSA update rule
           # Q[state, action] += alpha * (reward + gamma * np.max(Q[next_state]) - Q[state, action])
           Q[state, action] += alpha * (reward + gamma * Q[next_state,next_action] - Q[state, action])

           # move forward
           state = next_state
           action = next_action

       # Q_hist = np.vstack([Q_hist, Q.reshape(1, n_states, n_actions)])
   # return Q_hist
   return Q


def sarsa_lambda(env, n_episodes=100, alpha=0.1, gamma=1.0, epsilon=0.1, lambd=0.9, seed=None, verbose=True):
    """
    SARSA(lambda) 實作
    參數:
    - lambd: 資格跡衰減因子 (0~1)。
             0 = 普通 SARSA
             1 = 類似 Monte Carlo (直到結束才結算)
    """
    
    n_states = env.state_num
    n_actions = env.action_num
    
    # 1. 初始化 Q-table
    Q = np.zeros((n_states, n_actions))
    
    if seed is not None:
        np.random.seed(seed)

    def epsilon_greedy_policy(state):
        if np.random.rand() < epsilon:
            return env.action_sample()
        else:
            return np.argmax(Q[state])

    # 2. 開始訓練迴圈
    if verbose:
       iter_tool =  tqdm(range(n_episodes),desc='SARSA(𝜆)')
    else:
       iter_tool =  range(n_episodes)
    
    for idx in iter_tool:
        state = env.reset(idx)
        action = epsilon_greedy_policy(state)
        
        # [關鍵] 每個 Episode 開始時，資格跡 E 必須歸零
        E = np.zeros((n_states, n_actions))
        
        done = False

        while not done:
            next_state, reward, done = env.step(action)
            
            # 計算 TD Error (delta)
            # SARSA 是 On-policy，所以這裡必須根據「實際選的下一步」來算
            if done:
                # 如果結束了，未來的價值就是 0
                delta = reward - Q[state, action]
            else:
                next_action = epsilon_greedy_policy(next_state)
                delta = reward + gamma * Q[next_state, next_action] - Q[state, action]

            # [關鍵步驟 A] 增加當前狀態的資格跡 (Accumulating Trace)
            E[state, action] += 1
            
            # [關鍵步驟 B] 更新所有的 Q 值 (不僅僅是當前狀態)
            # Q(s,a) <- Q(s,a) + alpha * delta * E(s,a)
            # 這裡利用 numpy 的矩陣運算同時更新所有狀態，這是 SARSA(lambda) 的精髓
            Q += alpha * delta * E
            
            # [關鍵步驟 C] 衰減所有的資格跡
            # E(s,a) <- gamma * lambda * E(s,a)
            E *= gamma * lambd

            # 移動到下一步
            if not done:
                state = next_state
                action = next_action

    return Q


def backtest(test_returns, best_action, env, before, dt=1/252, rf=0.02):
    # for RL
    now_returns = deque(test_returns)
    e_t = deque([now_returns.popleft() for _ in range(before)],maxlen=before)
    M_t = np.sum(e_t)

    while True:
        st = encoding(e_t, M_t, env.bins_e, env.bins_M)
        action = best_action[st]
        
        if action == 1 or len(now_returns) == 0:
            if len(now_returns) > 0:
                next_r = now_returns.popleft()
                M_t += next_r
                reward = np.exp(M_t) * (1 + len(now_returns) * dt * rf) - 1
            else:
                reward = np.exp(M_t) - 1
            break
            
        else:
            next_r = now_returns.popleft()
            e_t.append(next_r)
            M_t += next_r
            
    return reward


def backtest2(test_returns, sell_action, dt=1/252,rf=0.02, ta=False):
    # for TA
    if sum(sell_action) == 0:
        sell_idx = len(test_returns)-1
    else:
        sell_idx = np.argmax(sell_action)
        if ta:
            sell_idx += 1
    reward = np.exp(np.sum(test_returns[:sell_idx+1])) * (1 + (len(test_returns)-sell_idx)*dt*rf) - 1
    return reward



In [2]:
df = pd.read_parquet(r'D:\MS113_LiuDL\論文\Project\台股\1101.TW.parquet')
# print(len(df))
train_df = df['close'].iloc[0:500].copy()
test_df = df['close'].iloc[500-1:600].copy()

log_returns =  np.log(train_df).diff().dropna()
test_returns = np.log(test_df).diff().dropna().tolist()

mu = log_returns.mean()
sigma = log_returns.std()
print(mu,sigma)

n_episodes = 10000
steps = 100
before = 3
dt = 1/252
rf = 0.02
alpha = 0.1
gamma = 1.0
epsilon = 0.2
lambd = 0.9

rounds = 1

sim_prices = bm_simulation_single(mu,sigma,state_n=n_episodes,steps=steps,vrt='',seed=None,keep_one=True)
sim_returns = np.diff(np.log(sim_prices),axis=1)



-0.0027142710678186065 0.03190366782002177


In [5]:
from joblib import Parallel, delayed, cpu_count

def Big_backtest(root_dir, stock_list, train_len, test_len, before=4, e_num_block=3, M_num_block=2, alpha=0.1, gamma=1.0, epsilon=0.1, lambd=0.3, rf=0.03):
    steps = test_len
    # before = 5
    # e_num_block = 7
    # M_num_block = 2
    times = 10
    n_episodes = min(times * e_num_block**before*M_num_block,2e5)


    dt = 1/252

    
    safe = False
    while safe == False:
        idx = np.random.randint(0,len(stock_list))
        df = pd.read_parquet(os.path.join(root_dir,stock_list[idx]),columns=['close'])
        if len(df) > (train_len + test_len):
            safe = True

    train_idx = np.random.randint(0,len(df)-train_len-test_len)
    train_df = df.iloc[train_idx:train_idx+train_len]
    test_df = df.iloc[train_idx+train_len-1:train_idx+train_len+test_len]
    
    close = df['close']
    df['MA'] = compute_MA(close)
    df['K'], df['D'] = compute_KD(close)
    df['KD'] = df['K'] - df['D']
    df['RSI'] = compute_RSI(close)
    _, _, df['MACD_hist'] = compute_MACD(close)
    
    train_returns =  np.log(train_df).diff().dropna().to_numpy().flatten()
    test_returns = np.log(test_df).diff().dropna().to_numpy().flatten()

    mu = train_returns.mean().item()
    sigma = train_returns.std().item()
    hurst = hurst_dfa(train_returns)
    # hurst = 0.8

    sim_prices = bm_simulation_single(mu,sigma,state_n=n_episodes,steps=steps,seed=None,keep_one=True)
    sim_returns = np.diff(np.log(sim_prices),axis=1)

    sim_prices2 = fbm_simulation_single(mu,sigma,hurst,state_n=n_episodes,steps=steps,seed=None,keep_one=True)
    sim_returns2 = np.diff(np.log(sim_prices2),axis=1)

    # RL
    env = SmartSelling_env(sim_returns, before, e_num_block, M_num_block)

    Q1 = q_learning(env, n_episodes, alpha, gamma, epsilon, verbose=False)
    best_action1 = np.argmax(Q1,axis=1)

    Q2 = sarsa(env, n_episodes, alpha, gamma, epsilon, verbose=False)
    best_action2 = np.argmax(Q2,axis=1)

    Q3 = sarsa_lambda(env, n_episodes, alpha, gamma, epsilon, lambd, verbose=False)
    best_action3 = np.argmax(Q3,axis=1)

    # reward計算
    rq = backtest(test_returns, best_action1, env, before, dt, rf)

    rs = backtest(test_returns, best_action2, env, before, dt, rf)

    rsl = backtest(test_returns, best_action3, env, before, dt, rf)

    action_BH = np.zeros((test_len,))
    rbh = backtest2(test_returns, action_BH)

    action_random = np.zeros((test_len,))
    action_random[np.random.randint(0,test_len)] = 1
    rr = backtest2(test_returns, action_random)

    action_ma = df['MA'].iloc[-test_len:].to_numpy()
    action_ma = (action_ma < 0).astype(int)
    rma = backtest2(test_returns, action_ma, ta=True)

    action_kd = df['KD'].iloc[-test_len:].to_numpy()
    action_kd = (action_kd < 0).astype(int)
    rkd = backtest2(test_returns, action_kd, ta=True)

    action_rsi = df['RSI'].iloc[-test_len:].to_numpy()
    action_rsi  = (action_rsi > 70).astype(int)
    rrsi = backtest2(test_returns, action_rsi, ta=True)

    action_macd = df['MACD_hist'].iloc[-test_len:].to_numpy()
    action_macd  = (action_macd < 0).astype(int)
    rmacd = backtest2(test_returns, action_macd, ta=True)

    action_rf = np.zeros((test_len,))
    action_rf[0] = 1
    rrf = backtest2(test_returns, action_rf)

    
    # RL+fbm
    env = SmartSelling_env(sim_returns2, before, e_num_block, M_num_block)

    Q1 = q_learning(env, n_episodes, alpha, gamma, epsilon, verbose=False)
    best_action1 = np.argmax(Q1,axis=1)

    Q2 = sarsa(env, n_episodes, alpha, gamma, epsilon, verbose=False)
    best_action2 = np.argmax(Q2,axis=1)

    Q3 = sarsa_lambda(env, n_episodes, alpha, gamma, epsilon, lambd, verbose=False)
    best_action3 = np.argmax(Q3,axis=1)

    # reward計算
    rq2 = backtest(test_returns, best_action1, env, before, dt, rf)

    rs2 = backtest(test_returns, best_action2, env, before, dt, rf)

    rsl2 = backtest(test_returns, best_action3, env, before, dt, rf)
    
    return rq, rs, rsl, rbh, rr, rma, rkd, rrsi, rmacd, rrf, rq2, rs2, rsl2



def Big_backtest2(root_dir, stock_list, train_len, test_len, before=4, e_num_block=3, M_num_block=2, alpha=0.1, gamma=1.0, epsilon=0.1, lambd=0.3, rf=0.03):
    steps = test_len
    # before = 5
    # e_num_block = 7
    # M_num_block = 2
    times = 10
    n_episodes = min(times * e_num_block**before*M_num_block,2e5)


    dt = 1/252

    
    safe = False
    while safe == False:
        idx = np.random.randint(0,len(stock_list))
        df = pd.read_parquet(os.path.join(root_dir,stock_list[idx]),columns=['close'])
        if len(df) > (train_len + test_len):
            safe = True

    train_idx = np.random.randint(0,len(df)-train_len-test_len)
    train_df = df.iloc[train_idx:train_idx+train_len]
    test_df = df.iloc[train_idx+train_len-1:train_idx+train_len+test_len]
    
    close = df['close']
    df['MA'] = compute_MA(close)
    df['K'], df['D'] = compute_KD(close)
    df['KD'] = df['K'] - df['D']
    df['RSI'] = compute_RSI(close)
    _, _, df['MACD_hist'] = compute_MACD(close)
    
    train_returns =  np.log(train_df).diff().dropna().to_numpy().flatten()
    test_returns = np.log(test_df).diff().dropna().to_numpy().flatten()

    mu = train_returns.mean().item()
    sigma = train_returns.std().item()
    hurst = hurst_dfa(train_returns)
    # hurst = 0.8

    sim_prices = bm_simulation_single(mu,sigma,state_n=n_episodes,steps=steps,seed=None,keep_one=True)
    sim_returns = np.diff(np.log(sim_prices),axis=1)

    fbm_times = 5
    sim_prices2 = fbm_simulation_single(mu/fbm_times,sigma/np.sqrt(fbm_times),hurst,state_n=n_episodes,steps=steps*fbm_times,seed=None,keep_one=True)
    sim_prices2 = sim_prices2[:,range(fbm_times-1,steps*fbm_times,fbm_times)]
    sim_returns2 = np.diff(np.log(sim_prices2),axis=1)

    # RL
    env = SmartSelling_env(sim_returns, before, e_num_block, M_num_block)

    Q1 = q_learning(env, n_episodes, alpha, gamma, epsilon, verbose=False)
    best_action1 = np.argmax(Q1,axis=1)

    Q2 = sarsa(env, n_episodes, alpha, gamma, epsilon, verbose=False)
    best_action2 = np.argmax(Q2,axis=1)

    Q3 = sarsa_lambda(env, n_episodes, alpha, gamma, epsilon, lambd, verbose=False)
    best_action3 = np.argmax(Q3,axis=1)

    # reward計算
    rq = backtest(test_returns, best_action1, env, before, dt, rf)

    rs = backtest(test_returns, best_action2, env, before, dt, rf)

    rsl = backtest(test_returns, best_action3, env, before, dt, rf)

    action_BH = np.zeros((test_len,))
    rbh = backtest2(test_returns, action_BH)

    action_random = np.zeros((test_len,))
    action_random[np.random.randint(0,test_len)] = 1
    rr = backtest2(test_returns, action_random)

    # TA
    # ... (前面的資料讀取部分保持不變) ...
    # ----------- 修改開始：優化 TA 賣出訊號 -----------

    # 1. MA (均線): ma5 由上到下穿越 ma20才賣
    ma_vals = df['MA'].iloc[-test_len:].to_numpy()
    ma_prev = df['MA'].shift(1).iloc[-test_len:].to_numpy()
    action_ma = ((ma_prev > 0) & (ma_vals <= 0)).astype(int)
    rma = backtest2(test_returns, action_ma, ta=True)
    
    # 2. KD (隨機指標): 優化為「高檔(>80)死亡交叉」
    # 邏輯：只有在 K > 80 的強勢區轉弱才賣，過濾掉盤整區的亂賣訊號
    k_vals = df['K'].iloc[-test_len:].to_numpy()
    kd_vals = df['KD'].iloc[-test_len:].to_numpy() # K - D
    # 條件：K < D (死叉) AND K > 80 (高檔)
    action_kd = ((kd_vals < 0) & (k_vals > 80)).astype(int)
    rkd = backtest2(test_returns, action_kd, ta=True)

    # 3. RSI (相對強弱): 優化為「跌破 70 賣出」
    # 邏輯：昨天 RSI > 70 (超買), 今天 RSI <= 70 (轉弱)
    rsi_vals = df['RSI'].iloc[-test_len:].to_numpy()
    rsi_prev = df['RSI'].shift(1).iloc[-test_len:].to_numpy() # 昨天的 RSI
    # 條件：昨天超買 且 今天跌破
    action_rsi = ((rsi_prev > 70) & (rsi_vals <= 70)).astype(int)
    rrsi = backtest2(test_returns, action_rsi, ta=True)

    # 4. MACD: 優化為「柱狀圖由正轉負」 (零軸下穿)
    # 邏輯：抓趨勢改變的第一個瞬間，而不是一直 < 0 就一直想賣
    macd_hist = df['MACD_hist'].iloc[-test_len:].to_numpy()
    macd_hist_prev = df['MACD_hist'].shift(1).iloc[-test_len:].to_numpy()
    # 條件：昨天紅柱(>0) 且 今天綠柱(<0)
    action_macd = ((macd_hist_prev > 0) & (macd_hist <= 0)).astype(int)
    rmacd = backtest2(test_returns, action_macd, ta=True)
    # ----------- 修改結束 -----------
    action_rf = np.zeros((test_len,))
    action_rf[0] = 1
    rrf = backtest2(test_returns, action_rf)

    
    # RL+fbm
    env = SmartSelling_env(sim_returns2, before, e_num_block, M_num_block)

    Q1 = q_learning(env, n_episodes, alpha, gamma, epsilon, verbose=False)
    best_action1 = np.argmax(Q1,axis=1)

    Q2 = sarsa(env, n_episodes, alpha, gamma, epsilon, verbose=False)
    best_action2 = np.argmax(Q2,axis=1)

    Q3 = sarsa_lambda(env, n_episodes, alpha, gamma, epsilon, lambd, verbose=False)
    best_action3 = np.argmax(Q3,axis=1)

    # reward計算
    rq2 = backtest(test_returns, best_action1, env, before, dt, rf)

    rs2 = backtest(test_returns, best_action2, env, before, dt, rf)

    rsl2 = backtest(test_returns, best_action3, env, before, dt, rf)
    
    return rq, rs, rsl, rbh, rr, rma, rkd, rrsi, rmacd, rrf, rq2, rs2, rsl2


#包含Heston
def Big_backtest3(root_dir, stock_list, train_len, test_len, before=4, e_num_block=3, M_num_block=2, alpha=0.1, gamma=1.0, epsilon=0.1, lambd=0.3, rf=0.03):
    steps = test_len
    # before = 5
    # e_num_block = 7
    # M_num_block = 2
    times = 10
    n_episodes = min(times * e_num_block**before*M_num_block,2e5)


    dt = 1/252

    
    safe = False
    while safe == False:
        idx = np.random.randint(0,len(stock_list))
        df = pd.read_parquet(os.path.join(root_dir,stock_list[idx]),columns=['close'])
        if len(df) > (train_len + test_len):
            safe = True

    train_idx = np.random.randint(0,len(df)-train_len-test_len)
    train_df = df.iloc[train_idx:train_idx+train_len]
    test_df = df.iloc[train_idx+train_len-1:train_idx+train_len+test_len]
    
    close = df['close']
    df['MA'] = compute_MA(close)
    df['K'], df['D'] = compute_KD(close)
    df['KD'] = df['K'] - df['D']
    df['RSI'] = compute_RSI(close)
    _, _, df['MACD_hist'] = compute_MACD(close)
    
    train_returns =  np.log(train_df).diff().dropna().to_numpy().flatten()
    test_returns = np.log(test_df).diff().dropna().to_numpy().flatten()

    mu = train_returns.mean().item()
    sigma = train_returns.std().item()
    hurst = hurst_dfa(train_returns)
    # hurst = 0.8

    sim_prices = bm_simulation_single(mu,sigma,state_n=n_episodes,steps=steps,seed=None,keep_one=True)
    sim_returns = np.diff(np.log(sim_prices),axis=1)

    sim_prices2 = fbm_simulation_single(mu,sigma,hurst,state_n=n_episodes,steps=steps,seed=None,keep_one=True)
    sim_returns2 = np.diff(np.log(sim_prices2),axis=1)


    mu_h, kappa, gamma, vv, rho = hm_params_final(train_returns, dt=1/252, n_particles=888, n_iter=220, burn_in=120, prop_std=None, verbose=False)
    _, _, sim_returns3 = hm_single(mu_h, kappa, gamma, vv, rho, dt, N=n_episodes, steps=steps, keep_one=False)
    
    
    # RL
    env = SmartSelling_env(sim_returns, before, e_num_block, M_num_block)

    Q1 = q_learning(env, n_episodes, alpha, gamma, epsilon, verbose=False)
    best_action1 = np.argmax(Q1,axis=1)

    Q2 = sarsa(env, n_episodes, alpha, gamma, epsilon, verbose=False)
    best_action2 = np.argmax(Q2,axis=1)

    Q3 = sarsa_lambda(env, n_episodes, alpha, gamma, epsilon, lambd, verbose=False)
    best_action3 = np.argmax(Q3,axis=1)

    # reward計算
    rq = backtest(test_returns, best_action1, env, before, dt, rf)

    rs = backtest(test_returns, best_action2, env, before, dt, rf)

    rsl = backtest(test_returns, best_action3, env, before, dt, rf)

    action_BH = np.zeros((test_len,))
    rbh = backtest2(test_returns, action_BH)

    action_random = np.zeros((test_len,))
    action_random[np.random.randint(0,test_len)] = 1
    rr = backtest2(test_returns, action_random)

    # TA
    # ... (前面的資料讀取部分保持不變) ...
    # ----------- 修改開始：優化 TA 賣出訊號 -----------

    # 1. MA (均線): ma5 由上到下穿越 ma20才賣
    ma_vals = df['MA'].iloc[-test_len:].to_numpy()
    ma_prev = df['MA'].shift(1).iloc[-test_len:].to_numpy()
    action_ma = ((ma_prev > 0) & (ma_vals <= 0)).astype(int)
    rma = backtest2(test_returns, action_ma, ta=True)
    
    # 2. KD (隨機指標): 優化為「高檔(>80)死亡交叉」
    # 邏輯：只有在 K > 80 的強勢區轉弱才賣，過濾掉盤整區的亂賣訊號
    k_vals = df['K'].iloc[-test_len:].to_numpy()
    kd_vals = df['KD'].iloc[-test_len:].to_numpy() # K - D
    # 條件：K < D (死叉) AND K > 80 (高檔)
    action_kd = ((kd_vals < 0) & (k_vals > 80)).astype(int)
    rkd = backtest2(test_returns, action_kd, ta=True)

    # 3. RSI (相對強弱): 優化為「跌破 70 賣出」
    # 邏輯：昨天 RSI > 70 (超買), 今天 RSI <= 70 (轉弱)
    rsi_vals = df['RSI'].iloc[-test_len:].to_numpy()
    rsi_prev = df['RSI'].shift(1).iloc[-test_len:].to_numpy() # 昨天的 RSI
    # 條件：昨天超買 且 今天跌破
    action_rsi = ((rsi_prev > 70) & (rsi_vals <= 70)).astype(int)
    rrsi = backtest2(test_returns, action_rsi, ta=True)

    # 4. MACD: 優化為「柱狀圖由正轉負」 (零軸下穿)
    # 邏輯：抓趨勢改變的第一個瞬間，而不是一直 < 0 就一直想賣
    macd_hist = df['MACD_hist'].iloc[-test_len:].to_numpy()
    macd_hist_prev = df['MACD_hist'].shift(1).iloc[-test_len:].to_numpy()
    # 條件：昨天紅柱(>0) 且 今天綠柱(<0)
    action_macd = ((macd_hist_prev > 0) & (macd_hist <= 0)).astype(int)
    rmacd = backtest2(test_returns, action_macd, ta=True)
    # ----------- 修改結束 -----------
    action_rf = np.zeros((test_len,))
    action_rf[0] = 1
    rrf = backtest2(test_returns, action_rf)

    
    # RL+fbm
    env = SmartSelling_env(sim_returns2, before, e_num_block, M_num_block)

    Q1 = q_learning(env, n_episodes, alpha, gamma, epsilon, verbose=False)
    best_action1 = np.argmax(Q1,axis=1)

    Q2 = sarsa(env, n_episodes, alpha, gamma, epsilon, verbose=False)
    best_action2 = np.argmax(Q2,axis=1)

    Q3 = sarsa_lambda(env, n_episodes, alpha, gamma, epsilon, lambd, verbose=False)
    best_action3 = np.argmax(Q3,axis=1)

    # reward計算
    rq2 = backtest(test_returns, best_action1, env, before, dt, rf)

    rs2 = backtest(test_returns, best_action2, env, before, dt, rf)

    rsl2 = backtest(test_returns, best_action3, env, before, dt, rf)


    # RL+heston
    env = SmartSelling_env(sim_returns3, before, e_num_block, M_num_block)

    Q1 = q_learning(env, n_episodes, alpha, gamma, epsilon, verbose=False)
    best_action1 = np.argmax(Q1,axis=1)

    Q2 = sarsa(env, n_episodes, alpha, gamma, epsilon, verbose=False)
    best_action2 = np.argmax(Q2,axis=1)

    Q3 = sarsa_lambda(env, n_episodes, alpha, gamma, epsilon, lambd, verbose=False)
    best_action3 = np.argmax(Q3,axis=1)

    # reward計算
    rq3 = backtest(test_returns, best_action1, env, before, dt, rf)

    rs3 = backtest(test_returns, best_action2, env, before, dt, rf)

    rsl3 = backtest(test_returns, best_action3, env, before, dt, rf)
    
    
    return rq, rs, rsl, rbh, rr, rma, rkd, rrsi, rmacd, rrf, rq2, rs2, rsl2, rq3, rs3, rsl3

<span style='font-size:16px; color:;'>  
    
###  實驗1. 100天賣股
* $e_t$: 參考近$3$期的股票報酬，使用標準差法離散化為$5$個區塊
* $M_t$: 使用百分位數法離散為$9$個區塊
* $\alpha: 0.1$
* $\gamma: 0.85$
* $\epsilon: 0.2$
* $\lambda: 0.4$
* RL訓練次數為狀態數量的$10$倍
* 測試$100$天
* 訓練長度為測試的$7$倍
* 總共$960$次的測試

In [7]:
# 固定參數
N = 32*30
test_len = 100
params_set = set()
best_list = []
train_len = 700
alpha = 0.1
gamma = 0.85
epsilon = 0.2
lambd = 0.40
rl_params = (alpha,gamma,epsilon,lambd)
my_columns = ['QL','SARSA','SARSA(𝜆)','B&H','❓','MA','KD','RSI','MACD','💰','QLf','SARSAf','SARSA(𝜆)f']
my_columns2 = ['QL','SARSA','SARSA(𝜆)','QLf','SARSAf','SARSA(𝜆)f']
while True:
    before = 3
    e_num_block = 5
    M_num_block = 9
    if ((before,e_num_block,M_num_block) not in params_set) and (e_num_block**before*M_num_block < 1e4):
        params_set.add((before,e_num_block,M_num_block))
        break
    else:
        params_set.add((before,e_num_block,M_num_block))

root_dir = "D:\MS113_LiuDL\論文\Project\台股"
stock_list = os.listdir(root_dir)
reward_list = Parallel(n_jobs=cpu_count())(delayed(Big_backtest)(root_dir, stock_list, train_len, test_len, before, e_num_block, M_num_block, alpha, gamma, epsilon, lambd) for i in tqdm(range(N),desc=''))
reward_array = np.vstack(reward_list)

reward_df = pd.DataFrame(reward_array,columns=my_columns)
rank_df = reward_df.rank(
    axis = 1,
    ascending = False,
    method = 'min'
    )

best_one = rank_df.describe().loc['mean',my_columns2].argmin()
best_list.append(my_columns2[best_one])
dev = rank_df.describe().at['mean',my_columns2[best_one]] - rank_df.describe().at['mean','B&H']
dev2 = rank_df.describe().loc['mean',my_columns2].mean() - rank_df.describe().at['mean','B&H']

info0 = f'{my_columns2[best_one]}： ({dev:.2f},{dev2:.2f})| {(before, e_num_block, M_num_block)} = {e_num_block**before*M_num_block}, ({rl_params}), len={train_len/test_len:.0f}'

info0 = '# ' + info0
print(info0)
print('實際報酬比較')
print(reward_df.describe().map(lambda x: f'{x:.2%}').to_string())
print()
print('報酬排名比較')
print(rank_df.describe().map(lambda x: f'{x:.2f}').to_string())
print('-'*120)
    



100%|██████████| 960/960 [22:12<00:00,  1.39s/it]


# QLf： (-1.81,-1.57)| (3, 5, 9) = 1125, ((0.1, 0.85, 0.2, 0.4)), len=7
實際報酬比較
              QL      SARSA   SARSA(𝜆)        B&H          ❓         MA         KD        RSI       MACD          💰        QLf     SARSAf  SARSA(𝜆)f
count  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%
mean       3.17%      1.52%      2.81%      7.39%      4.52%      1.39%      0.91%      2.92%      1.48%      0.79%      1.85%      1.39%      2.83%
std       41.87%      7.80%     39.66%     50.98%     43.98%      6.77%      6.88%     21.89%      7.34%      2.95%      7.73%      7.93%     42.23%
min      -50.07%    -41.04%    -40.94%    -56.60%    -56.11%    -28.33%    -49.97%    -56.60%    -30.95%    -49.55%    -50.07%    -51.29%    -44.43%
25%        0.23%      0.06%      0.26%     -9.76%     -6.25%     -1.03%     -2.11%     -6.47%     -1.16%     -0.23%      0.21%     -0.02%      0.10%
50%        1.91%      1.96% 

<span style='font-size:16px; color:;'>  

### 結果分析-實驗1(100天)
以下歸納RL方法在本實驗中的核心優勢。  

**1. 壓倒性的中位數表現（勝率的保證）**  
中位數（Median）代表了策略在「大多數情況下」的表現。這是 RL 方法最強大的護城河。
* RL 完勝 B&H：  
    * B&H 中位數僅 $0.75\%$，這意味著有一半的時間，買入持有策略的獲利微乎其微，甚至不如銀行利息（$0.79\%$）。
    * RL 方法（如 QLf, SARSA($\lambda$)）中位數高達 $2.07\% ～ 2.40\%$。
    * 結論：在隨機的市場波動中，RL 策略能穩定地創造出 比 B&H 高出近 $3$ 倍 的中位數回報。這證明了 RL 能夠有效過濾無效波動，在有獲利時果斷落袋。

**2. 排名穩定性居冠**  
從報酬排名（Rank）來看，RL 是表現最穩定的策略群體。  
* 平均排名第一：
    * QLf (使用 fBM 訓練) 取得了全場最佳的平均排名 $4.98$，是唯一平均排名進入前 $5$ 名的策略。
    * 相比之下，B&H 平均排名為 $6.79$，MA 為 $7.44$，KD 為 $7.96$。

* 穩定的前段班：
    * RL 方法的排名標準差（Std）顯著低於 B&H，且中位數排名穩定維持在 $5$。這代表 RL 策略極少墊底，無論市場好壞，它都能維持在所有策略的前半段。

<span style='font-size:16px; color:;'>  
    
###  實驗2. 30天賣出
* $e_t$: 參考近$3$期的股票報酬，使用標準差法離散化為$5$個區塊
* $M_t$: 使用百分位數法離散為$9$個區塊
* $\alpha: 0.1$
* $\gamma: 0.85$
* $\epsilon: 0.2$
* $\lambda: 0.4$
* RL訓練次數為狀態數量的$10$倍
* 測試$30$天
* 訓練長度為測試的$7$倍
* 總共$960$次的測試

In [17]:
# 固定參數
N = 32*30
test_len = 30
params_set = set()
best_list = []
train_times = 7
train_len = test_len * train_times
alpha = 0.1
gamma = 0.85
epsilon = 0.2
lambd = 0.40
rl_params = (alpha,gamma,epsilon,lambd)
my_columns = ['QL','SARSA','SARSA(𝜆)','B&H','❓','MA','KD','RSI','MACD','💰','QLf','SARSAf','SARSA(𝜆)f']
my_columns2 = ['QL','SARSA','SARSA(𝜆)','QLf','SARSAf','SARSA(𝜆)f']
while True:
    before = 3
    e_num_block = 5
    M_num_block = 9
    if ((before,e_num_block,M_num_block) not in params_set) and (e_num_block**before*M_num_block < 1e4):
        params_set.add((before,e_num_block,M_num_block))
        break
    else:
        params_set.add((before,e_num_block,M_num_block))

root_dir = "D:\MS113_LiuDL\論文\Project\台股"
stock_list = os.listdir(root_dir)
reward_list = Parallel(n_jobs=cpu_count())(delayed(Big_backtest)(root_dir, stock_list, train_len, test_len, before, e_num_block, M_num_block, alpha, gamma, epsilon, lambd) for i in tqdm(range(N),desc=''))
reward_array = np.vstack(reward_list)

reward_df = pd.DataFrame(reward_array,columns=my_columns)
rank_df = reward_df.rank(
    axis = 1,
    ascending = False,
    method = 'min'
    )

best_one = rank_df.describe().loc['mean',my_columns2].argmin()
best_list.append(my_columns2[best_one])
dev = rank_df.describe().at['mean',my_columns2[best_one]] - rank_df.describe().at['mean','B&H']
dev2 = rank_df.describe().loc['mean',my_columns2].mean() - rank_df.describe().at['mean','B&H']

info0 = f'{my_columns2[best_one]}： ({dev:.2f},{dev2:.2f})| {(before, e_num_block, M_num_block)} = {e_num_block**before*M_num_block}, ({rl_params}), len={train_len/test_len:.0f}'

info0 = '# ' + info0
print(info0)
print('實際報酬比較')
print(reward_df.describe().map(lambda x: f'{x:.2%}').to_string())
print()
print('報酬排名比較')
print(rank_df.describe().map(lambda x: f'{x:.2f}').to_string())
print('-'*120)
    



100%|██████████| 960/960 [14:49<00:00,  1.08it/s]


# SARSA(𝜆)： (-0.86,-0.81)| (3, 5, 9) = 1125, ((0.1, 0.85, 0.2, 0.4)), len=7
實際報酬比較
              QL      SARSA   SARSA(𝜆)        B&H          ❓         MA         KD        RSI       MACD          💰        QLf     SARSAf  SARSA(𝜆)f
count  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%
mean       0.47%      0.64%      0.68%      1.89%      1.36%      0.76%      0.48%      1.54%      0.78%      0.25%      0.33%      0.31%      0.71%
std        7.89%      8.00%      8.88%     16.31%     11.92%      7.19%      5.34%     15.16%      6.35%      2.30%      8.18%      8.54%      8.66%
min      -40.82%    -40.26%    -40.26%    -40.81%    -40.81%    -50.69%    -49.29%    -40.81%    -50.69%     -9.79%    -40.82%    -40.82%    -40.26%
25%       -0.83%     -0.68%     -1.28%     -6.08%     -3.40%     -1.63%     -1.38%     -5.38%     -1.32%     -0.69%     -1.15%     -1.12%     -1.22%
50%        1.03%      1

<span style='font-size:16px; color:;'>  

### 結果分析-實驗2(30天)
即使賣出天數調降至30天(短天期)，RL賣出方法的中位數優勢依舊存在，排名的平均還比B&H少了$0.81$，也就是說RL是穩穩賺錢的好方法。

<span style='font-size:16px; color:;'>  
    
###  實驗3. 100天賣出，強化版TA
#### 參數設定和實驗1相同，但是強化了技術分析(TA)的賣出邏輯，賣出方法調整如下：
* MA：慢線 $<$ 快線 $\Rightarrow$ 慢線由上往下穿越快線死亡交叉
* RSI： rsi $>$ 70 $\Rightarrow$ 昨天 > 70 且 今天 < 70
* KD： K值 $ < $ D值 $\Rightarrow$ 高檔鈍化(K值>80)後死叉
* MACD： macd柱狀體 $<$ 0  $\Rightarrow$ 昨天hist $>$ 0 且 今天 hist $<$ 0

In [18]:
# 強化版TA 固定參數
N = 32*30
test_len = 100
params_set = set()
best_list = []
train_len = 700
alpha = 0.1
gamma = 0.85
epsilon = 0.2
lambd = 0.40
rl_params = (alpha,gamma,epsilon,lambd)
my_columns = ['QL','SARSA','SARSA(𝜆)','B&H','❓','MA','KD','RSI','MACD','💰','QLf','SARSAf','SARSA(𝜆)f']
my_columns2 = ['QL','SARSA','SARSA(𝜆)','QLf','SARSAf','SARSA(𝜆)f']
while True:
    before = 3
    e_num_block = 5
    M_num_block = 9
    if ((before,e_num_block,M_num_block) not in params_set) and (e_num_block**before*M_num_block < 1e4):
        params_set.add((before,e_num_block,M_num_block))
        break
    else:
        params_set.add((before,e_num_block,M_num_block))

root_dir = "D:\MS113_LiuDL\論文\Project\台股"
stock_list = os.listdir(root_dir)
reward_list = Parallel(n_jobs=cpu_count())(delayed(Big_backtest2)(root_dir, stock_list, train_len, test_len, before, e_num_block, M_num_block, alpha, gamma, epsilon, lambd) for i in tqdm(range(N),desc=''))
reward_array = np.vstack(reward_list)

reward_df = pd.DataFrame(reward_array,columns=my_columns)
rank_df = reward_df.rank(
    axis = 1,
    ascending = False,
    method = 'min'
    )

best_one = rank_df.describe().loc['mean',my_columns2].argmin()
best_list.append(my_columns2[best_one])
dev = rank_df.describe().at['mean',my_columns2[best_one]] - rank_df.describe().at['mean','B&H']
dev2 = rank_df.describe().loc['mean',my_columns2].mean() - rank_df.describe().at['mean','B&H']

info0 = f'{my_columns2[best_one]}： ({dev:.2f},{dev2:.2f})| {(before, e_num_block, M_num_block)} = {e_num_block**before*M_num_block}, ({rl_params}), len={train_len/test_len:.0f}'

info0 = '# ' + info0
print(info0)
print('實際報酬比較')
print(reward_df.describe().map(lambda x: f'{x:.2%}').to_string())
print()
print('報酬排名比較')
print(rank_df.describe().map(lambda x: f'{x:.2f}').to_string())
print('-'*120)
    



100%|██████████| 960/960 [22:04<00:00,  1.38s/it]


# SARSA(𝜆)： (-1.27,-1.18)| (3, 5, 9) = 1125, ((0.1, 0.85, 0.2, 0.4)), len=7
實際報酬比較
              QL      SARSA   SARSA(𝜆)        B&H          ❓         MA         KD        RSI       MACD          💰        QLf     SARSAf  SARSA(𝜆)f
count  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%
mean       1.78%      2.05%      2.26%      5.40%      3.37%      2.61%      3.45%      3.45%      2.05%      0.85%      1.96%      1.89%      2.15%
std        8.66%      8.72%      9.82%     28.66%     19.08%     15.72%     23.17%     22.33%     14.90%      2.37%      9.25%      8.22%      9.62%
min      -50.38%    -42.94%    -48.12%    -53.95%    -47.12%    -38.81%    -52.45%    -53.95%    -44.80%     -9.29%    -50.38%    -41.24%    -47.85%
25%        0.07%      0.07%      0.70%    -10.80%     -6.01%     -4.22%     -6.72%     -7.05%     -4.31%     -0.20%      0.12%      0.08%      0.28%
50%        1.95%      2

<span style='font-size:16px; color:;'>  

#### 另一組參數設計
* $e_t$: 參考近$7$期的股票報酬，使用標準差法離散化為$2$個區塊
* $M_t$: 使用百分位數法離散為$2$個區塊
* $\alpha: 0.1$
* $\gamma: 0.85$
* $\epsilon: 0.25$
* $\lambda: 0.25$
* RL訓練次數為狀態數量的$10$倍
* 測試$100$天
* 訓練長度為測試的$6$倍
* 總共$960$次的測試

In [10]:
# 強化版TA 固定參數
N = 32*30
test_len = 100
params_set = set()
best_list = []
train_len = 600
alpha = 0.1
gamma = 0.85
epsilon = 0.25
lambd = 0.25
rl_params = (alpha,gamma,epsilon,lambd)
my_columns = ['QL','SARSA','SARSA(𝜆)','B&H','❓','MA','KD','RSI','MACD','💰','QLf','SARSAf','SARSA(𝜆)f']
my_columns2 = ['QL','SARSA','SARSA(𝜆)','QLf','SARSAf','SARSA(𝜆)f']
while True:
    before = 7
    e_num_block = 2
    M_num_block = 2
    if ((before,e_num_block,M_num_block) not in params_set) and (e_num_block**before*M_num_block < 1e4):
        params_set.add((before,e_num_block,M_num_block))
        break
    else:
        params_set.add((before,e_num_block,M_num_block))

root_dir = "D:\MS113_LiuDL\論文\Project\台股"
stock_list = os.listdir(root_dir)
reward_list = Parallel(n_jobs=cpu_count())(delayed(Big_backtest2)(root_dir, stock_list, train_len, test_len, before, e_num_block, M_num_block, alpha, gamma, epsilon, lambd) for i in tqdm(range(N),desc=''))
reward_array = np.vstack(reward_list)

reward_df = pd.DataFrame(reward_array,columns=my_columns)
rank_df = reward_df.rank(
    axis = 1,
    ascending = False,
    method = 'min'
    )

best_one = rank_df.describe().loc['mean',my_columns2].argmin()
best_list.append(my_columns2[best_one])
dev = rank_df.describe().at['mean',my_columns2[best_one]] - rank_df.describe().at['mean','B&H']
dev2 = rank_df.describe().loc['mean',my_columns2].mean() - rank_df.describe().at['mean','B&H']

info0 = f'{my_columns2[best_one]}： ({dev:.2f},{dev2:.2f})| {(before, e_num_block, M_num_block)} = {e_num_block**before*M_num_block}, ({rl_params}), len={train_len/test_len:.0f}'

info0 = '# ' + info0
print(info0)
print('實際報酬比較')
print(reward_df.describe().map(lambda x: f'{x:.2%}').to_string())
print()
print('報酬排名比較')
print(rank_df.describe().map(lambda x: f'{x:.2f}').to_string())
print('-'*120)
    
# 7. QLf： (-1.65,-1.41)| (7, 2, 2) = 256, ((0.09, 0.83, 0.27, 0.24)), len=6



100%|██████████| 960/960 [05:09<00:00,  3.10it/s]


# SARSA： (-1.53,-1.33)| (7, 2, 2) = 256, ((0.1, 0.85, 0.25, 0.25)), len=6
實際報酬比較
              QL      SARSA   SARSA(𝜆)        B&H          ❓         MA         KD        RSI       MACD          💰        QLf     SARSAf  SARSA(𝜆)f
count  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%  96000.00%
mean       2.21%      2.43%      2.41%      7.17%      3.85%      2.97%      4.03%      4.73%      2.40%      0.82%      2.13%      2.31%      2.12%
std       10.38%     10.19%     10.34%     37.40%     21.39%     15.70%     25.10%     26.90%     13.62%      2.48%     10.35%     10.11%     10.51%
min      -57.84%    -57.84%    -57.84%    -73.40%    -69.43%    -49.02%    -71.78%    -71.47%    -49.02%     -9.24%    -57.84%    -57.60%    -56.68%
25%       -0.18%     -0.11%     -0.03%     -8.95%     -5.78%     -3.35%     -6.47%     -6.32%     -3.09%     -0.15%     -0.22%     -0.02%     -0.20%
50%        2.61%      2.6

<span style='font-size:16px; color:;'>  

### 結果分析-實驗3(100天，強化TA)
強化TA比原本的簡單TA好很多，平均報酬還有排名都到了能夠與隨機賣出一較高下的地步，但是RL的中位數以及排名仍然大幅輾壓這些基本、技術分析的方法。因此使用技術分析，不如長期持有或是用這裡的RL方法。

<span style='font-size:16px; color:;'>  
    
###  實驗4. 隨機RL參數
為了近一步確認RL方法的穩定性，實驗四會使用隨機的參數來做強化學習agent的訓練。
* $e_t$: 參考近($1～7$)期的股票報酬，使用標準差法離散化為($2～10$)個區塊
* $M_t$: 使用百分位數法離散為($2～10$)個區塊
* $\alpha: 0.01～0.3$
* $\gamma: 0.7～1.0$
* $\epsilon: 0.03～0.3$
* $\lambda: 0.05～0.95$
* $100$天內賣出
* 訓練長度為測試的($1～10$)倍
* 每組參數測試96次，有20組
* 強化版TA

In [3]:
N = 32*3
test_len = 100
params_set = set()
best_list = []
my_columns = ['QL','SARSA','SARSA(𝜆)','B&H','❓','MA','KD','RSI','MACD','💰','QLf','SARSAf','SARSA(𝜆)f']
my_columns2 = ['QL','SARSA','SARSA(𝜆)','QLf','SARSAf','SARSA(𝜆)f']
big_reward_list = []
N2 = 20
for idx in range(N2):
    train_len = int(test_len * random.randint(1,10))
    alpha = round(random.uniform(0.01,0.30),2)
    gamma = round(random.uniform(0.70,1.0),2)
    epsilon = round(random.uniform(0.03,0.3),2)
    lambd = round(random.uniform(0.05,0.95),2)
    rl_params = (alpha,gamma,epsilon,lambd)
    while True:
        before = random.randint(1,7)
        e_num_block = random.randint(2,10)
        M_num_block = random.randint(2,10)
        if ((before,e_num_block,M_num_block) not in params_set) and (e_num_block**before*M_num_block < 5e3):
            params_set.add((before,e_num_block,M_num_block))
            break
        else:
            params_set.add((before,e_num_block,M_num_block))

    root_dir = "D:\MS113_LiuDL\論文\Project\台股"
    stock_list = os.listdir(root_dir)
    reward_list = Parallel(n_jobs=cpu_count())(delayed(Big_backtest2)(root_dir, stock_list, train_len, test_len, before, e_num_block, M_num_block, alpha, gamma, epsilon, lambd) for i in tqdm(range(N),desc=''))
    big_reward_list.extend(reward_list)
    reward_array = np.vstack(reward_list)

    reward_df = pd.DataFrame(reward_array,columns=my_columns)
    row_maxes = reward_df.max(axis=1)
    Q1 = row_maxes.quantile(0.25)
    Q3 = row_maxes.quantile(0.75)
    IQR = Q3 - Q1
    limit = Q3 + 30.0 * IQR
    clean_reward_df = reward_df[row_maxes <= limit]
    clean_rank_df = clean_reward_df.rank(
        axis = 1,
        ascending = False,
        method = 'min'
        )
    
    best_one = clean_rank_df.describe().loc['mean',my_columns2].argmin()
    best_list.append(my_columns2[best_one])
    dev = clean_rank_df.describe().at['mean',my_columns2[best_one]] - clean_rank_df.describe().at['mean','B&H']
    dev2 = clean_rank_df.describe().loc['mean',my_columns2].mean() - clean_rank_df.describe().at['mean','B&H']
    
    info0 = f'{idx}. {my_columns2[best_one]}： ({dev:.2f},{dev2:.2f})| {(before, e_num_block, M_num_block)} = {e_num_block**before*M_num_block}, ({rl_params}), len={train_len/test_len:.0f}'

    info0 = '# ' + info0
    print(info0)
    print(clean_reward_df.describe().map(lambda x: f'{x:.2%}').to_string())
    print()
    print(clean_rank_df.describe().map(lambda x: f'{x:.2f}').to_string())
    print('-'*120)


print()
print("+"*150)
n = len(best_list)
counter = Counter(best_list)
ratio_dic = {k:v/n for k,v in counter.items()}
ratio_dic = dict(sorted(ratio_dic.items(),key=lambda x:-x[1]))
for k,v in ratio_dic.items():
    print(f'{k}: {v:.2%}')


big_reward_array = np.vstack(big_reward_list)
big_reward_df = pd.DataFrame(big_reward_array,columns=my_columns)
row_maxes = big_reward_df.max(axis=1)
Q1 = row_maxes.quantile(0.25)
Q3 = row_maxes.quantile(0.75)
IQR = Q3 - Q1
limit = Q3 + 30.0 * IQR
clean_df = big_reward_df[row_maxes <= limit]

clean_rank_df = clean_df.rank(
    axis = 1,
    ascending = False,
    method = 'min'
    )

print(f"{N2}組隨機參數綜合結果如下")
print('實際報酬')
print(clean_df.describe().loc['mean':].map(lambda x: f'{x:.2%}').to_string())
print('報酬排行')
print(clean_rank_df.describe().map(lambda x: f'{x:.2f}').to_string())



100%|██████████| 96/96 [00:12<00:00,  7.67it/s]


# 0. SARSAf： (-1.09,-0.88)| (1, 10, 7) = 70, ((0.21, 0.97, 0.07, 0.55)), len=7
             QL     SARSA  SARSA(𝜆)       B&H         ❓        MA        KD       RSI      MACD         💰       QLf    SARSAf SARSA(𝜆)f
count  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%
mean      2.67%     2.64%     2.91%     7.65%     4.53%     3.31%     4.24%     2.75%     3.26%     0.41%     2.59%     2.87%     1.87%
std       5.61%     4.60%     6.63%    33.58%    20.68%    16.20%    18.58%    17.26%    10.81%     2.13%     5.32%     4.47%     6.64%
min     -26.22%   -20.91%   -21.83%   -33.23%   -49.97%   -46.16%   -37.36%   -38.31%   -46.65%    -8.88%   -26.22%   -20.91%   -23.66%
25%       1.23%     1.17%     1.43%    -9.56%    -6.34%    -1.92%    -4.77%    -5.25%    -1.34%    -0.32%     1.02%     1.09%     0.94%
50%       2.29%     2.30%     3.05%     1.38%    -0.16%     1.69%     0.68%     0.03%     1.75%     0.55%

100%|██████████| 96/96 [00:16<00:00,  5.73it/s]


# 1. QL： (-0.28,-0.00)| (3, 3, 10) = 270, ((0.29, 0.79, 0.28, 0.86)), len=10
             QL     SARSA  SARSA(𝜆)       B&H         ❓        MA        KD       RSI      MACD         💰       QLf    SARSAf SARSA(𝜆)f
count  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%
mean      0.83%     1.07%     0.84%    10.44%     6.19%     3.01%     4.51%     4.22%     2.02%     1.00%     1.10%     1.20%     1.29%
std      10.43%     7.71%     7.08%    33.29%    23.47%    15.98%    23.31%    20.57%    14.01%     2.72%     9.72%     6.96%     6.52%
min     -56.31%   -38.02%   -23.12%   -56.31%   -54.34%   -28.00%   -49.71%   -36.73%   -23.12%    -4.99%   -56.31%   -24.04%   -24.04%
25%      -0.05%    -0.47%    -1.13%    -7.89%    -4.84%    -5.49%    -5.72%    -5.80%    -4.89%    -0.20%    -0.43%    -0.42%    -1.53%
50%       2.07%     1.55%     1.07%     3.60%     0.55%     1.17%     0.06%    -0.38%    -0.97%     0.79%  

100%|██████████| 96/96 [00:53<00:00,  1.78it/s]


...

# 17. QL： (-1.07,-0.55)| (6, 2, 7) = 448, ((0.25, 0.96, 0.07, 0.76)), len=5
             QL     SARSA  SARSA(𝜆)       B&H         ❓        MA        KD       RSI      MACD         💰       QLf    SARSAf SARSA(𝜆)f
count  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%
mean      2.44%     1.93%     1.99%     5.38%     3.07%     1.97%     1.59%     2.33%     0.62%     0.81%     1.69%     1.85%     1.68%
std       4.89%     5.71%     4.61%    27.58%    14.44%     9.18%    17.48%    18.15%     8.02%     2.22%     6.54%     4.96%     6.17%
min     -19.98%   -19.98%    -9.67%   -27.53%   -23.24%   -21.29%   -38.96%   -36.35%   -25.78%    -6.26%   -27.54%   -19.66%   -23.62%
25%       0.60%     0.05%    -0.29%   -10.41%    -3.78%    -2.77%    -8.87%    -8.40%    -3.40%    -0.02%     0.04%     0.27%    -1.03%
50%       2.41%     2.00%     1.86%     0.31%     0.71%     0.68%     0.96%     1.06%     0.88%     0.79%   

100%|██████████| 96/96 [03:09<00:00,  1.98s/it]


# 18. SARSA(𝜆)f： (-1.19,-0.65)| (3, 8, 5) = 2560, ((0.2, 0.87, 0.15, 0.35)), len=5
             QL     SARSA  SARSA(𝜆)       B&H         ❓        MA        KD       RSI      MACD         💰       QLf    SARSAf SARSA(𝜆)f
count  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%
mean      0.23%     1.04%     1.85%     3.04%     1.99%     0.33%     1.02%     1.80%     1.05%     0.96%     0.97%     1.28%     1.85%
std       9.56%     6.84%     7.41%    25.69%    19.77%    16.90%    22.68%    20.96%    14.26%     2.21%     8.59%     6.75%     8.34%
min     -51.76%   -34.59%   -42.50%   -74.77%   -56.49%   -46.81%   -74.77%   -46.51%   -48.38%    -6.02%   -51.76%   -35.12%   -42.50%
25%       0.03%     0.03%     1.00%    -7.85%    -5.20%    -4.44%    -5.91%    -6.97%    -3.82%     0.14%     0.09%     0.15%     0.78%
50%       1.94%     2.21%     2.46%     0.98%     0.87%     0.40%     0.62%     0.82%     0.50%     0

100%|██████████| 96/96 [00:00<00:00, 130.34it/s]


# 19. SARSAf： (-2.22,-1.53)| (2, 2, 2) = 8, ((0.28, 0.79, 0.14, 0.42)), len=8
             QL     SARSA  SARSA(𝜆)       B&H         ❓        MA        KD       RSI      MACD         💰       QLf    SARSAf SARSA(𝜆)f
count  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%  9600.00%
mean      2.72%     1.55%     1.56%     4.82%     0.46%     3.18%     3.01%     4.29%     2.40%     1.17%     2.12%     2.51%     1.94%
std       8.74%     8.24%     8.84%    25.20%    18.56%    16.14%    19.28%    21.06%    15.46%     3.41%     8.57%     7.44%     8.81%
min     -40.89%   -40.89%   -40.89%   -41.42%   -48.64%   -25.80%   -30.41%   -30.41%   -34.98%    -6.20%   -40.89%   -24.17%   -40.89%
25%       0.14%    -0.08%    -0.84%    -9.86%    -8.29%    -4.65%    -6.25%    -6.93%    -3.66%     0.27%     0.09%     0.14%     0.34%
50%       2.49%     1.79%     1.79%    -0.60%    -1.37%     0.67%    -0.62%    -1.12%     1.02%     0.79% 

<span style='font-size:16px; color:;'>  

### 結果分析-實驗4(隨機RL參數)
**RL穩定的表現**  
即使使用隨機的RL參數，其穩定的表現依舊存在。
* 中位數：RL方法約為2%報酬，勝過其他基本和技術分析的方法，顯示大部分的情況能夠賺取更大的報酬。
* 25%分位數：RL方法報酬多數大於0%，表示即使遇到不景氣的市場情況，也有相當的抗跌能力。
* 報酬排行：排名上位者皆為RL方法(排名5.57～5.89)，是勝率的領先者
* 結論：想要穩定的賺取報酬，RL是最佳的選擇。

<span style='font-size:16px; color:;'>  
    
###  實驗5. 隨機RL參數 + Heston model
#### 新增Heston model模擬股價(-h結尾)的RL方法。總計測試100個隨機RL參數組。

In [4]:
# 加入Heston model
N = 32*5
test_len = 100
params_set = set()
best_list = []
my_columns = ['QL','SARSA','SARSA(𝜆)','B&H','❓','MA','KD','RSI','MACD','💰','QLf','SARSAf','SARSA(𝜆)f','QLh','SARSAh','SARSA(𝜆)h']
my_columns2 = ['QL','SARSA','SARSA(𝜆)','QLf','SARSAf','SARSA(𝜆)f','QLh','SARSAh','SARSA(𝜆)h']
big_reward_list = []
N2 = 100
for idx in range(N2):
    train_len = int(test_len * random.randint(1,10))
    alpha = round(random.uniform(0.01,0.30),2)
    gamma = round(random.uniform(0.70,1.0),2)
    epsilon = round(random.uniform(0.03,0.3),2)
    lambd = round(random.uniform(0.05,0.95),2)
    rl_params = (alpha,gamma,epsilon,lambd)
    while True:
        before = random.randint(1,7)
        e_num_block = random.randint(2,10)
        M_num_block = random.randint(2,10)
        if ((before,e_num_block,M_num_block) not in params_set) and (e_num_block**before*M_num_block < 5e3):
            params_set.add((before,e_num_block,M_num_block))
            break
        else:
            params_set.add((before,e_num_block,M_num_block))

    root_dir = "D:\MS113_LiuDL\論文\Project\台股"
    stock_list = os.listdir(root_dir)
    reward_list = Parallel(n_jobs=cpu_count())(delayed(Big_backtest3)(root_dir, stock_list, train_len, test_len, before, e_num_block, M_num_block, alpha, gamma, epsilon, lambd) for i in tqdm(range(N),desc=''))
    big_reward_list.extend(reward_list)
    reward_array = np.vstack(reward_list)

    reward_df = pd.DataFrame(reward_array,columns=my_columns)
    row_maxes = reward_df.max(axis=1)
    Q1 = row_maxes.quantile(0.25)
    Q3 = row_maxes.quantile(0.75)
    IQR = Q3 - Q1
    limit = Q3 + 30.0 * IQR
    clean_reward_df = reward_df[row_maxes <= limit]
    clean_rank_df = clean_reward_df.rank(
        axis = 1,
        ascending = False,
        method = 'min'
        )
    
    best_one = clean_rank_df.describe().loc['mean',my_columns2].argmin()
    best_list.append(my_columns2[best_one])
    dev = clean_rank_df.describe().at['mean',my_columns2[best_one]] - clean_rank_df.describe().at['mean','B&H']
    dev2 = clean_rank_df.describe().loc['mean',my_columns2].mean() - clean_rank_df.describe().at['mean','B&H']
    
    info0 = f'{idx}. {my_columns2[best_one]}： ({dev:.2f},{dev2:.2f})| {(before, e_num_block, M_num_block)} = {e_num_block**before*M_num_block}, ({rl_params}), len={train_len/test_len:.0f}'

    info0 = '# ' + info0
    print(info0)
    print(clean_reward_df.describe().loc['mean':].map(lambda x: f'{x:.1%}').to_string())
    print()
    print(clean_rank_df.describe().map(lambda x: f'{x:.2f}').to_string())
    print('-'*120)


print()
print("+"*150)
n = len(best_list)
counter = Counter(best_list)
ratio_dic = {k:v/n for k,v in counter.items()}
ratio_dic = dict(sorted(ratio_dic.items(),key=lambda x:-x[1]))
for k,v in ratio_dic.items():
    print(f'{k}: {v:.2%}')


big_reward_array = np.vstack(big_reward_list)
big_reward_df = pd.DataFrame(big_reward_array,columns=my_columns)
row_maxes = big_reward_df.max(axis=1)
Q1 = row_maxes.quantile(0.25)
Q3 = row_maxes.quantile(0.75)
IQR = Q3 - Q1
limit = Q3 + 30.0 * IQR
clean_df = big_reward_df[row_maxes <= limit]

clean_rank_df = clean_df.rank(
    axis = 1,
    ascending = False,
    method = 'min'
    )

print(f"{N2}組隨機參數綜合結果如下")
print('實際報酬')
print(clean_df.describe().loc['mean':].map(lambda x: f'{x:.1%}').to_string())
print('報酬排行')
print(clean_rank_df.describe().loc['mean':].map(lambda x: f'{x:.2f}').to_string())

# 99. QLf： (-4.11,-3.79)| (2, 9, 10) = 810, ((0.05, 0.84, 0.17, 0.55)), len=7
# 96. QL： (-3.20,-2.63)| (3, 5, 9) = 1125, ((0.27, 0.74, 0.15, 0.72)), len=9
# 70. QL： (-2.94,-2.44)| (7, 2, 9) = 1152, ((0.14, 0.97, 0.27, 0.44)), len=4
# 67. SARSA(𝜆)f： (-3.04,-2.80)| (4, 3, 10) = 810, ((0.02, 0.84, 0.11, 0.47)), len=9
# 31. QLf： (-3.27,-2.90)| (4, 3, 8) = 648, ((0.07, 0.87, 0.26, 0.65)), len=3
# 29. QL： (-3.49,-2.98)| (4, 2, 9) = 144, ((0.1, 0.79, 0.27, 0.86)), len=6
# 18. SARSA(𝜆)h： (-3.17,-2.89)| (2, 9, 9) = 729, ((0.05, 0.96, 0.3, 0.95)), len=6
# 16. QLh： (-3.61,-3.24)| (3, 5, 10) = 1250, ((0.15, 0.76, 0.12, 0.63)), len=6
# 12. QL： (-3.46,-3.21)| (2, 4, 6) = 96, ((0.01, 0.76, 0.2, 0.47)), len=1
# 11. SARSAh： (-3.31,-3.07)| (3, 5, 8) = 1000, ((0.03, 0.91, 0.24, 0.55)), len=4
# 1. SARSA(𝜆)： (-3.48,-3.08)| (4, 2, 8) = 128, ((0.15, 0.86, 0.17, 0.73)), len=1



100%|██████████| 160/160 [04:26<00:00,  1.66s/it]


# 0. SARSA： (-1.43,-1.16)| (1, 7, 7) = 49, ((0.22, 0.72, 0.17, 0.17)), len=9
          QL   SARSA SARSA(𝜆)     B&H       ❓      MA      KD     RSI    MACD      💰     QLf  SARSAf SARSA(𝜆)f     QLh  SARSAh SARSA(𝜆)h
mean    2.1%    1.8%     1.9%    5.8%    2.4%    1.9%    3.8%    4.1%    0.5%   1.0%    1.2%    1.7%      1.9%    1.5%    1.3%      1.4%
std     6.1%    6.1%     5.9%   22.2%   16.1%   12.3%   19.4%   18.4%    8.4%   2.3%    7.9%    6.8%      6.3%    7.9%    7.9%      6.8%
min   -33.2%  -33.2%   -34.2%  -56.6%  -56.6%  -36.4%  -56.6%  -41.8%  -34.3%  -5.5%  -56.6%  -34.2%    -27.0%  -56.6%  -56.6%    -32.5%
25%     0.6%    0.6%     0.5%   -6.9%   -4.1%   -2.4%   -5.3%   -4.5%   -2.7%  -0.0%    0.3%    0.7%      0.5%    0.4%    0.4%      0.2%
50%     2.3%    2.1%     2.1%    3.1%    1.2%    0.7%    0.3%    1.0%    0.5%   0.8%    2.0%    2.1%      2.1%    1.9%    1.8%      1.8%
75%     4.6%    4.3%     4.4%   15.2%    7.2%    5.4%    8.8%    7.0%    4.1%   1.8%    3.9%    4.4% 

100%|██████████| 160/160 [00:48<00:00,  3.31it/s]


# 1. SARSA(𝜆)： (-3.48,-3.08)| (4, 2, 8) = 128, ((0.15, 0.86, 0.17, 0.73)), len=1
          QL   SARSA SARSA(𝜆)     B&H       ❓      MA      KD     RSI    MACD      💰     QLf  SARSAf SARSA(𝜆)f     QLh  SARSAh SARSA(𝜆)h
mean    2.5%    2.5%     2.7%    3.1%    3.2%    1.7%    2.4%    1.9%    1.0%   1.1%    2.1%    2.8%      2.4%    3.1%    3.6%      2.6%
std    10.1%   11.0%     8.8%   25.8%   18.5%   11.7%   16.9%   18.5%   12.0%   2.5%   11.3%    9.4%     10.5%   15.3%   13.9%     11.7%
min   -71.8%  -71.8%   -32.8%  -71.8%  -49.5%  -33.2%  -37.2%  -57.2%  -71.8%  -5.7%  -71.8%  -57.2%    -57.2%  -71.8%  -31.4%    -57.2%
25%     0.6%    0.5%     0.6%  -11.2%   -5.5%   -3.0%   -5.5%   -7.3%   -3.1%  -0.2%    0.5%    0.5%      0.4%    0.3%    0.3%      0.3%
50%     2.4%    2.3%     2.4%   -1.0%    0.8%    0.7%    0.3%   -0.0%    1.1%   0.8%    2.4%    2.3%      2.1%    2.2%    2.3%      2.2%
75%     5.0%    4.4%     4.4%   12.0%    8.4%    4.0%    6.5%    7.6%    5.0%   1.6%    5.2%    4

100%|██████████| 160/160 [00:23<00:00,  6.93it/s]


...

# 97. QL： (-3.04,-2.76)| (1, 3, 10) = 30, ((0.17, 0.97, 0.19, 0.19)), len=10
          QL   SARSA SARSA(𝜆)     B&H       ❓      MA      KD     RSI    MACD      💰     QLf  SARSAf SARSA(𝜆)f     QLh  SARSAh SARSA(𝜆)h
mean    1.0%    1.3%     1.0%    4.7%    1.5%    1.9%    2.1%    2.3%    0.7%   1.0%    2.4%    2.0%      2.6%    0.7%    1.8%      1.4%
std    10.9%    8.8%    10.2%   34.1%   18.8%   15.1%   28.5%   27.8%   11.7%   2.7%   20.1%   20.3%     19.5%    9.6%   20.3%     10.5%
min   -64.2%  -55.1%   -64.2%  -64.2%  -55.0%  -65.0%  -72.6%  -76.1%  -66.2%  -6.2%  -64.2%  -64.2%    -55.1%  -64.2%  -64.2%    -49.6%
25%     0.4%    0.4%     0.6%  -12.0%   -7.6%   -3.5%   -8.3%   -9.1%   -2.9%  -0.3%    0.3%    0.2%      0.4%    0.4%    0.2%      0.4%
50%     2.2%    1.9%     2.0%   -0.7%    0.5%    0.4%   -0.9%    0.2%    0.3%   0.8%    1.9%    1.8%      1.8%    1.7%    1.8%      1.9%
75%     4.8%    4.5%     4.5%   10.8%    5.2%    5.6%    7.0%    8.8%    5.5%   1.5%    4.5%    4.2% 

100%|██████████| 160/160 [03:53<00:00,  1.46s/it]


# 98. SARSA(𝜆)： (-1.79,-1.69)| (1, 3, 3) = 9, ((0.02, 0.86, 0.06, 0.1)), len=8
          QL   SARSA SARSA(𝜆)     B&H       ❓      MA      KD     RSI    MACD      💰     QLf  SARSAf SARSA(𝜆)f     QLh  SARSAh SARSA(𝜆)h
mean    2.5%    2.6%     2.1%    7.0%    3.7%    2.3%    4.1%    3.2%    2.3%   0.6%    2.8%    2.4%      2.8%    2.7%    2.1%      2.6%
std     8.4%   11.6%     7.9%   31.9%   18.8%   12.3%   24.2%   24.3%   10.2%   2.3%   12.3%    8.3%     12.1%    7.2%    6.8%      9.8%
min   -57.9%  -57.9%   -57.9%  -57.9%  -36.6%  -41.9%  -57.9%  -57.9%  -32.9%  -7.1%  -57.9%  -57.9%    -57.9%  -24.2%  -22.1%    -57.9%
25%     0.2%   -0.0%     0.0%   -9.3%   -4.7%   -3.5%   -6.8%   -7.6%   -2.7%  -0.2%    0.2%    0.1%     -0.3%    0.2%   -0.1%      0.0%
50%     1.7%    1.6%     1.6%    0.6%    1.6%    0.8%    1.8%    0.7%    1.2%   0.8%    2.1%    1.7%      1.8%    1.8%    2.1%      1.8%
75%     4.2%    3.7%     4.3%   13.3%    9.2%    8.0%   11.2%    8.5%    7.2%   1.5%    4.6%    4.5

100%|██████████| 160/160 [05:59<00:00,  2.25s/it]


# 99. QLf： (-4.11,-3.79)| (2, 9, 10) = 810, ((0.05, 0.84, 0.17, 0.55)), len=7
          QL   SARSA SARSA(𝜆)     B&H       ❓      MA      KD     RSI    MACD      💰     QLf  SARSAf SARSA(𝜆)f     QLh  SARSAh SARSA(𝜆)h
mean    1.2%    1.2%     1.2%    1.6%    1.2%    0.6%    1.3%    0.7%    1.6%   0.8%    1.2%    1.4%      1.3%    1.0%    0.8%      1.1%
std     9.7%    7.5%     8.2%   25.5%   15.6%   11.0%   20.6%   15.7%   13.3%   2.4%    9.2%    6.1%      6.5%   10.1%    8.1%      8.1%
min   -71.0%  -36.9%   -36.9%  -71.0%  -32.9%  -27.1%  -71.0%  -40.5%  -30.4%  -9.2%  -71.0%  -31.0%    -28.9%  -71.0%  -40.5%    -40.5%
25%     0.5%    0.3%     0.5%  -11.7%   -6.2%   -5.3%   -7.4%   -7.7%   -5.3%  -0.2%    0.5%    0.2%      0.1%    0.5%    0.3%      0.6%
50%     1.9%    1.8%     1.9%   -2.0%   -0.1%   -0.1%    0.3%    0.2%    0.1%   0.8%    1.8%    1.7%      1.7%    1.8%    1.7%      1.9%
75%     4.1%    3.9%     4.0%   11.5%    6.6%    5.1%    7.4%    9.1%    4.3%   1.5%    4.0%    3.9%

<span style='font-size:16px; color:;'>  

### 結果分析-實驗5(隨機RL參數+Heston)
#### RL方法比較
* 三種模擬來源的RL方法差距並不大，其中以使用布朗運動(BM)的Q-learning(QL)方法略勝一籌，平均排名在$6$以內，獲得最佳排名的次數占($23\%$)

<span style='font-size:16px; color:;'>  
    
## 六. 結論與未來展望  <a id="結論與未來展望"></a>
本文使用三種RL方法，Q-learning、SARSA、SARSA(𝜆)來自動選擇最佳的賣股時機點，並且使用時間序列的拔靴法進行回測。與基本和技術分析方法比較，在中位數、Q1報酬以及報酬排名上大幅領先，表示RL能夠穩定的賺取報酬，取得了漂亮的成就。  
未來可以嘗試reward改成別的績效指標，例如Sharpe或是Utility函數。或是不要一次賣出股票，動作可以換成一次賣出一部份的股票，讓操作更符合實務。


<span style='font-size:16px; color:;'>

## 七.參考文獻 <a id="參考文獻"></a>

Corazza, M., & Sangalli, A. (2015). Q-learning and SARSA: A comparison 
between two intelligent stochastic control approaches for financial trading (Working 
Paper No. 15/WP/2015). Department of Economics, Ca’ Foscari University of 
Venice.  

Efron, B., & Tibshirani, R. J. (1993). An introduction to the bootstrap. 
New York, NY: Chapman & Hall/CRC.  

Haruvi, Y. (2017). Applying bootstrap methods to time series and regression models. 
M.Sc. Seminar in Statistics, Tel Aviv University. 
    


Powell, W. B. (2022). Reinforcement learning and stochastic optimization: 
A unified framework for sequential decisions. Hoboken, NJ: John Wiley & Sons.

